# 02 — Dados, feedback e exploração

**DATA SCIENCE KNOWLEDGE BASE**  
**Área:** `09_recommender_systems` — Sistemas de Recomendação  
**Capítulo:** 02 de 16  

> Execute as células em sequência, de cima para baixo, em um kernel reiniciado. Use o **Guia Técnico** e a **Cheat Sheet** da seção Materiais complementares como referências para este capítulo.

## Objetivos de aprendizagem
- Investigar a estrutura das interações
- Verificar duplicatas, tipos, valores ausentes e distribuição

## Materiais complementares

- [Guia Técnico](reference_materials/02_dados_interacoes.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_02_dados_interacoes.pdf)

**Navegação:** os links são relativos à pasta deste notebook. A versão editorial desses PDFs será revalidada após a consolidação científica dos notebooks.

## Fundamentação conceitual
Em recomendação, a tabela de eventos geralmente possui $(u,i,r,t)$, onde $r$ representa nota e $t$ representa instante. Contagens ajudam a identificar itens com suporte estatístico baixo.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## Qualidade dos dados

In [2]:
print('Nulos:',ratings.isna().sum().to_dict())
print('Duplicatas usuário–filme:',ratings.duplicated(['userId','movieId']).sum())
print('Filmes sem título correspondente:',(~ratings.movieId.isin(movies.movieId)).sum())

Nulos: {'userId': 0, 'movieId': 0, 'rating': 0, 'timestamp': 0}
Duplicatas usuário–filme: 0
Filmes sem título correspondente: 0


**Interpretação.** Verifique premissas antes de gerar recomendações.

## Cauda longa e notas

In [3]:
activity=ratings.groupby('userId').size(); popularity=ratings.groupby('movieId').size()
print('Atividade usuários:',activity.quantile([.1,.5,.9]).to_dict())
print('Notas por filme:',popularity.quantile([.1,.5,.9]).to_dict())
print(ratings.rating.value_counts().sort_index())

Atividade usuários: {0.1: 24.0, 0.5: 70.5, 0.9: 400.30000000000007}
Notas por filme: {0.1: 1.0, 0.5: 3.0, 0.9: 27.0}
rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64


**Interpretação.** Poucos itens recebem muitas avaliações; muitos aparecem raramente.

## Dimensão temporal

In [4]:
t=pd.to_datetime(ratings.timestamp,unit='s',utc=True)
print('Período:',t.min().date(),t.max().date())

Período: 1996-03-29 2018-09-24


**Interpretação.** Partições aleatórias podem misturar eventos futuros e passados.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** feedback explícito e implícito. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **feedback explícito e implícito** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **feedback explícito e implícito** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — Qualidade do evento e recorte temporal

O registro `(usuário, item, nota, instante)` descreve uma interação **observada**, não uma oportunidade de exposição. Audite duplicatas, nulos, faixa de notas e ordem temporal. A ausência de interações fora de uma janela pode indicar falta de exposição, não desinteresse.

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [5]:
# Estatísticas de integridade, sem modificar os dados originais
checks = {
    'chaves_duplicadas': int(ratings.duplicated(['userId','movieId']).sum()),
    'ids_sem_catalogo': int((~ratings.movieId.isin(movies.movieId)).sum()),
    'datas_invalidas': int(pd.to_datetime(ratings.timestamp, unit='s', utc=True, errors='coerce').isna().sum()),
}
print(checks)
assert ratings.rating.between(0.5,5).all()

{'chaves_duplicadas': 0, 'ids_sem_catalogo': 0, 'datas_invalidas': 0}


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

## Limitações e boas práticas
Dados públicos de avaliações estão sujeitos a seleção, exposição e popularidade.

## Exercícios
Identifique o filme mais avaliado e procure seu título.

## Soluções comentadas
`mid=ratings.movieId.value_counts().idxmax(); movies.loc[movies.movieId.eq(mid),"title"]`.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados do capítulo em `outputs/`.


In [6]:
# EP_OUTPUTS_09_V1
# Exportação após os experimentos. Executar este notebook desde o início (Run All).
# Arquivos com prefixo do capítulo evitam colisões com outras saídas.
from pathlib import Path
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
ativ = ratings.groupby("userId").size().rename("n_interacoes").reset_index()
ativ.to_csv(out / "02_atividade_usuarios.csv", index=False)
cont = ratings.groupby("movieId").size().rename("n_avaliacoes").reset_index()
cont.merge(movies[["movieId", "title"]], on="movieId", how="left").to_csv(
    out / "02_popularidade_itens.csv", index=False)
print('Outputs do capítulo 02 salvos em:', out.resolve())


Outputs do capítulo 02 salvos em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/09_recommender_systems/outputs
